In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 96.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=72e8139c244ef74b8fb0b6ea620d1015068adf840f9819191380d1a22670c829
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [3]:
# Step 1: Helper function to generate quantum random bits

def quantum_random_bit():
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)

    simulator = BasicSimulator()
    result = simulator.run(qc, shots=1).result()
    counts = result.get_counts()

    bit = list(counts.keys())[0]
    return int(bit)

In [4]:
# Step 2: Sender generates random bits and bases

number_of_bits = 50

sender_bits = []
sender_bases = []

for i in range(number_of_bits):
    sender_bits.append(quantum_random_bit())
    sender_bases.append(quantum_random_bit())

print("Sender bits: ", sender_bits)
print("Sender bases:", sender_bases)

Sender bits:  [1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1]
Sender bases: [0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1]


In [6]:
# Step 3: Sender prepares qubits

prepared_qubits = []

for i in range(number_of_bits):

    qc = QuantumCircuit(1, 1)

    # Encode bit
    if sender_bits[i] == 1:
        qc.x(0)

    # Encode basis
    if sender_bases[i] == 1:
        qc.h(0)

    prepared_qubits.append(qc)

print("Sender prepared qubits.")


Sender prepared qubits.


In [7]:
# Step 4: Attacker intercepts and measures the qubits

attacker_bases = []
qubits_after_attack = []

simulator = BasicSimulator()

for i in range(number_of_bits):

    qc = prepared_qubits[i].copy()

    # Attacker chooses random basis
    attacker_basis = quantum_random_bit()
    attacker_bases.append(attacker_basis)

    # If attacker uses X basis, apply Hadamard before measuring
    if attacker_basis == 1:
        qc.h(0)

    qc.measure(0, 0)

    result = simulator.run(qc, shots=1).result()
    counts = result.get_counts()
    attacker_bit = int(list(counts.keys())[0])

    # Attacker prepares a new qubit to send to receiver
    new_qc = QuantumCircuit(1, 1)

    if attacker_bit == 1:
        new_qc.x(0)

    if attacker_basis == 1:
        new_qc.h(0)

    qubits_after_attack.append(new_qc)

print("Attacker measured and resent the qubits.")

Attacker measured and resent the qubits.


In [8]:
# Step 5: Receiver generates random bases

receiver_bases = []

for i in range(number_of_bits):
    receiver_bases.append(quantum_random_bit())

print("Receiver bases:", receiver_bases)

Receiver bases: [0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1]


In [9]:
# Step 6: Receiver measures the qubits after attack

receiver_results = []

simulator = BasicSimulator()

for i in range(number_of_bits):

    qc = qubits_after_attack[i].copy()

    if receiver_bases[i] == 1:
        qc.h(0)

    qc.measure(0, 0)

    result = simulator.run(qc, shots=1).result()
    counts = result.get_counts()
    measured_bit = int(list(counts.keys())[0])

    receiver_results.append(measured_bit)

print("Receiver results:", receiver_results)

Receiver results: [1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1]


In [10]:
# Step 7: Compare bases and calculate error rate

matching_sender_bits = []
matching_receiver_bits = []

for i in range(number_of_bits):

    if sender_bases[i] == receiver_bases[i]:
        matching_sender_bits.append(sender_bits[i])
        matching_receiver_bits.append(receiver_results[i])

errors = 0

for i in range(len(matching_sender_bits)):
    if matching_sender_bits[i] != matching_receiver_bits[i]:
        errors += 1

error_rate = errors / len(matching_sender_bits)

print("Sender matching bits:  ", matching_sender_bits)
print("Receiver matching bits:", matching_receiver_bits)
print("Number of errors:", errors)
print("Error rate:", error_rate)

Sender matching bits:   [1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1]
Receiver matching bits: [1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1]
Number of errors: 8
Error rate: 0.2857142857142857


In [11]:
# Step 8: Detect whether an attacker is present

threshold = 0.2

if error_rate > threshold:
    print("Attack detected!")
else:
    print("No attack detected.")

print("Threshold:", threshold)
print("Actual error rate:", error_rate)

Attack detected!
Threshold: 0.2
Actual error rate: 0.2857142857142857
